# A+B+Diversifier Sleeves — Analytics Notebook

This notebook reconstructs the ensemble strategy and produces:
- Cumulative returns
- Key metrics (Total Return, CAGR, Sharpe, Sortino, Max DD, Win Rate, Profit Factor, PSR, DSR)
- Drawdown over time
- Daily returns distribution
- Monthly returns heatmap
- Rolling Sharpe (252 days)
- Rolling volatility (252 days)


In [ ]:
import sys
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add project root to path so Live package is importable.
PROJECT_ROOT = Path.cwd().parent if 'Live' in str(Path.cwd()) else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Live.live.data_feed import fetch_panel
from Live.live.portfolio import build_sleeve_returns, SleeveConfig

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('Imports OK')

## 1. Load Strategy A / B returns and sleeve prices

In [ ]:
# Paths to pre-computed core returns (generated by research scripts).
CORE_A_CSV = PROJECT_ROOT / '_rsi_rotation_compare_strategy_A.csv'
CORE_B_CSV = PROJECT_ROOT / '_rsi_rotation_compare_strategy_B.csv'

ret_a = pd.read_csv(CORE_A_CSV, index_col='date', parse_dates=['date'])['strategy_return']
ret_b = pd.read_csv(CORE_B_CSV, index_col='date', parse_dates=['date'])['strategy_return']

common_index = ret_a.index.intersection(ret_b.index)
ret_a = ret_a.loc[common_index]
ret_b = ret_b.loc[common_index]

# Sleeve ETF universe.
sleeve_tickers = ['SPY','TLT','IEF','GLD','PDBC','KMLM','DBMF','VIXM','SH','BIL','VIXY','PSQ','^VIX']
end = ret_a.index.max().date()
start = ret_a.index.min().date() - timedelta(days=365)

prices = fetch_panel(sleeve_tickers, start, end, prefer_alpaca=False)
prices = prices.rename(columns={'^VIX': 'VIX'})

print(f'Core A/B dates: {ret_a.index.min().date()} -> {ret_a.index.max().date()}')
print(f'Price panel shape: {prices.shape}')

## 2. Build ensemble return series

In [ ]:
config = SleeveConfig()
sleeve_rets = build_sleeve_returns(prices)
sleeve_rets = sleeve_rets.reindex(common_index)

# Daily ensemble net return (fixed sleeve weights).
ensemble_ret = (
    config.weight_a * ret_a
    + config.weight_b * ret_b
    + config.weight_rates * sleeve_rets['rates']
    + config.weight_bear * sleeve_rets['bear']
    + config.weight_cta * sleeve_rets['cta']
)
ensemble_ret = ensemble_ret.dropna()

cum = (1 + ensemble_ret).cumprod()
print(f'Ensemble dates: {ensemble_ret.index.min().date()} -> {ensemble_ret.index.max().date()}')


## 3. Key metrics

In [ ]:
def probabilistic_sharpe_ratio(returns, target_sharpe=0.0, ann_factor=252.0):
    r = returns.dropna()
    T = len(r)
    if T < 30:
        return np.nan
    mean, std = r.mean(), r.std(ddof=1)
    if std == 0:
        return np.nan
    sr_daily = mean / std
    sr = sr_daily * np.sqrt(ann_factor)
    skew = r.skew()
    kurt = r.kurtosis() + 3.0
    denom_sq = 1.0 - skew * sr_daily + ((kurt - 1.0) / 4.0) * (sr_daily ** 2)
    if denom_sq <= 0:
        return np.nan
    z = (sr - target_sharpe) * np.sqrt(T - 1) / np.sqrt(denom_sq)
    return float(stats.norm.cdf(z))

def deflated_sharpe_ratio(returns, n_trials=10, annualized_sharpe=None):
    """Bailey & López de Prado deflated SR approximation."""
    r = returns.dropna()
    T = len(r)
    if T < 30 or n_trials <= 0:
        return np.nan
    if annualized_sharpe is None:
        annualized_sharpe = r.mean() / r.std(ddof=1) * np.sqrt(252)
    # Expected max SR under null = sqrt of trials * E[chi].
    expected_max = np.sqrt(-np.log(1 - 0.5 ** (1.0 / n_trials)) / np.log(4))
    if annualized_sharpe <= expected_max:
        return 0.0
    sr_daily = annualized_sharpe / np.sqrt(252)
    skew = r.skew()
    kurt = r.kurtosis() + 3.0
    denom_sq = 1.0 - skew * sr_daily + ((kurt - 1.0) / 4.0) * (sr_daily ** 2)
    if denom_sq <= 0:
        return np.nan
    z = (annualized_sharpe - expected_max) * np.sqrt(T - 1) / np.sqrt(denom_sq)
    return float(stats.norm.cdf(z))

def metrics_table(returns, n_trials=10):
    r = returns.dropna()
    years = len(r) / 252.0
    cum = (1 + r).cumprod()
    total = cum.iloc[-1] - 1
    cagr = (cum.iloc[-1]) ** (1 / years) - 1 if years > 0 else 0.0
    ann_vol = r.std() * np.sqrt(252)
    sharpe = (r.mean() * 252) / ann_vol if ann_vol > 0 else 0.0
    downside = r[r < 0]
    sortino = (r.mean() * 252) / (downside.std() * np.sqrt(252)) if len(downside) > 1 else 0.0
    peak = cum.cummax()
    max_dd = ((cum - peak) / peak).min()
    win_rate = (r > 0).mean()
    gross_profit = r[r > 0].sum()
    gross_loss = -r[r < 0].sum()
    profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.nan
    psr = probabilistic_sharpe_ratio(r)
    dsr = deflated_sharpe_ratio(r, n_trials=n_trials, annualized_sharpe=sharpe)
    return pd.Series({
        'Total Return': f'{total*100:.2f}%',
        'CAGR': f'{cagr*100:.2f}%',
        'Ann Vol': f'{ann_vol*100:.2f}%',
        'Sharpe': f'{sharpe:.2f}',
        'Sortino': f'{sortino:.2f}',
        'Max DD': f'{max_dd*100:.2f}%',
        'Win Rate': f'{win_rate*100:.2f}%',
        'Profit Factor': f'{profit_factor:.2f}',
        'PSR': f'{psr:.2%}',
        'DSR': f'{dsr:.2%}',
    })

metrics = metrics_table(ensemble_ret)
display(metrics.to_frame('Value'))

## 4. Cumulative returns

In [ ]:
fig, ax = plt.subplots()
cum.plot(ax=ax, label='A+B+Diversifier Sleeves', linewidth=1.5)
ax.axhline(1.0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('Cumulative Returns')
ax.set_ylabel('Growth of $1')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Drawdown over time

In [ ]:
peak = cum.cummax()
dd = (cum - peak) / peak

fig, ax = plt.subplots()
dd.plot(ax=ax, color='crimson', linewidth=1.0)
ax.fill_between(dd.index, dd, 0, color='crimson', alpha=0.3)
ax.set_title('Drawdown Over Time')
ax.set_ylabel('Drawdown')
ax.set_ylim(dd.min() * 1.1, 0.02)
plt.tight_layout()
plt.show()
print(f'Max drawdown: {dd.min()*100:.2f}% on {dd.idxmin().date()}')

## 6. Daily returns distribution

In [ ]:
fig, ax = plt.subplots()
ensemble_ret.hist(bins=100, ax=ax, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(ensemble_ret.mean(), color='red', linestyle='--', label=f'Mean={ensemble_ret.mean()*100:.3f}%')
ax.set_title('Daily Returns Distribution')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Skew: {ensemble_ret.skew():.2f}, Kurtosis: {ensemble_ret.kurtosis():.2f}')

## 7. Monthly returns heatmap

In [ ]:
monthly = ensemble_ret.resample('ME').apply(lambda x: (1 + x).prod() - 1) * 100
monthly_table = monthly.to_frame('ret').assign(
    year=monthly.index.year,
    month=monthly.index.month
).pivot(index='year', columns='month', values='ret')

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(monthly_table, annot=True, fmt='.1f', cmap='RdYlGn', center=0, ax=ax, linewidths=0.5)
ax.set_title('Monthly Returns Heatmap (%)')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 8. Rolling Sharpe (252 days)

In [ ]:
rolling_sharpe = (ensemble_ret.rolling(252).mean() * 252) / (ensemble_ret.rolling(252).std() * np.sqrt(252))

fig, ax = plt.subplots()
rolling_sharpe.plot(ax=ax, color='green', linewidth=1.0)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('Rolling 252-Day Sharpe Ratio')
ax.set_ylabel('Sharpe')
plt.tight_layout()
plt.show()
print(f'Median rolling Sharpe: {rolling_sharpe.median():.2f}')

## 9. Rolling volatility (252 days)

In [ ]:
rolling_vol = ensemble_ret.rolling(252).std() * np.sqrt(252) * 100

fig, ax = plt.subplots()
rolling_vol.plot(ax=ax, color='purple', linewidth=1.0)
ax.axhline(rolling_vol.mean(), color='black', linestyle='--', linewidth=0.8, label=f'Mean={rolling_vol.mean():.1f}%')
ax.set_title('Rolling 252-Day Annualized Volatility')
ax.set_ylabel('Volatility (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Component sleeve cumulative returns

In [ ]:
components = pd.DataFrame({
    'A': ret_a.reindex(ensemble_ret.index),
    'B': ret_b.reindex(ensemble_ret.index),
    'Rates': sleeve_rets['rates'].reindex(ensemble_ret.index),
    'Bear': sleeve_rets['bear'].reindex(ensemble_ret.index),
    'CTA': sleeve_rets['cta'].reindex(ensemble_ret.index),
})
cum_components = (1 + components).cumprod()

fig, ax = plt.subplots(figsize=(14, 7))
cum_components.plot(ax=ax, linewidth=1.2)
ax.set_title('Cumulative Returns by Sleeve Component')
ax.set_ylabel('Growth of $1')
ax.legend()
plt.tight_layout()
plt.show()